# 🧬 BindCORE Studio

### Per-residue **LIP** and **MoRF** binding prediction from sequence


## How to start

1. **Runtime → Change runtime type → T4 GPU**, then **Connect**.
2. **Click ▶ on the single cell below.** Everything is cloned and installed
   automatically by an animated checklist — runs once (~8–12 min).
3. The **Studio** opens underneath: paste a sequence, **Run BindCORE**, and watch the
   per-protein 4-stage checklist. Explore the interactive plots when it's done.

<p style="color:red">Note: Due to Google Colab's resource limitations as well as IDPFold2's constraints, longer proteins may take significantly longer to process and might lead to timeouts. </p>


In [ ]:
#@title 🧬 **BindCORE Studio** — click the ▶ button to start { display-mode: "form" }
#@markdown All repositories are cloned automatically. An animated checklist installs the stack once (~8–12 min), then the studio appears below.
import os, sys, subprocess, glob, shutil, time, threading, select
from IPython.display import display, update_display, HTML

# Public repositories — cloned automatically (pre-coded; no user input needed).
BINDCORE_URL    = "https://github.com/nbuton/BindCORE"
IDPFOLD2_URL    = "https://github.com/Junjie-Zhu/IDPFold2"
ENSEMBLEMDP_URL = "https://github.com/nbuton/EnsembleMDP"
CG2ALL_SPEC     = "git+https://github.com/huhlim/cg2all@cuda-12"
IDPFOLD2_CKPT_URL = "https://zenodo.org/records/18239596/files/IDPFold2_ema_0.999_260114.pth?download=1"

# TODO: ADD ZOTERO/HF LINK WITH MODEL WEIGHTS
BINDCORE_MODELS_URL = "https://zenodo.org/records/20752230/files/models.zip?download=1"

CONTENT    = "/content"
BINDCORE   = f"{CONTENT}/BindCORE"
IDPFOLD2   = f"{CONTENT}/IDPFold2"
CKPT_DIR   = f"{CONTENT}/checkpoints"
IDP_CKPT   = f"{CKPT_DIR}/IDPFold2_ema_0.999_260114.pth"
WORK       = f"{CONTENT}/bindcore_work"
CG2ALL_ENV = f"{CONTENT}/cg2all_env"
CG2ALL_BIN = f"{CG2ALL_ENV}/bin/convert_cg2all"
DGL_INDEX  = "https://data.dgl.ai/wheels/torch-2.3/repo.html"
for _d in (CKPT_DIR, WORK):
    os.makedirs(_d, exist_ok=True)
UVPIP = "uv pip install --system"

# ---- shared CLI-style animation primitives (used by setup AND the run) ----
_SPIN = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"
_UIFONT = "Inter, system-ui, -apple-system, Roboto, Helvetica, Arial, sans-serif"
_SHIM_CSS = ("<style>@keyframes bcsh{0%,100%{opacity:.45}50%{opacity:1}}"
             ".bcsh{animation:bcsh 1.25s ease-in-out infinite}</style>")


class _Installer:
    """A main-thread animated task list. tick() is called from the subprocess poll
    loop so the spinner advances without any background thread (which Colab does not
    reliably flush from)."""
    def __init__(self, title):
        self.title = title; self.steps = []; self.f = 0; self.did = "bc_setup_ui"
        display(HTML(self._html()), display_id=self.did)

    def begin(self, label):
        self.steps.append([label, "run", time.time(), 0.0]); self._r(); return self.steps[-1]

    def end(self, rec, ok):
        rec[3] = time.time() - rec[2]; rec[1] = "ok" if ok else "err"; self._r()

    def tick(self):
        self.f += 1; self._r()

    def close(self, title=None):
        if title:
            self.title = title
        self._r()

    def _row(self, rec):
        lbl, st, t0, dur = rec
        if st == "run":
            ic = f"<span style='color:#6c5ce7;font-size:15px'>{_SPIN[self.f % len(_SPIN)]}</span>"
            nm = f"<span class='bcsh' style='color:#3b2f63;font-weight:600'>{lbl}</span>"
            tm = f"{time.time() - t0:.0f}s"
        elif st == "ok":
            ic = "<span style='color:#27ae60;font-size:14px'>✓</span>"
            nm = f"<span style='color:#1f2430'>{lbl}</span>"; tm = f"{dur:.0f}s"
        else:
            ic = "<span style='color:#e8633a;font-size:14px'>✗</span>"
            nm = f"<span style='color:#e8633a;font-weight:600'>{lbl}</span>"; tm = ""
        return (f"<div style='display:flex;align-items:center;gap:11px;padding:4px 0'>"
                f"<span style='width:16px;text-align:center'>{ic}</span>"
                f"<span style='flex:1'>{nm}</span>"
                f"<span style='color:#9aa3b2;font-size:12px;"
                f"font-variant-numeric:tabular-nums'>{tm}</span></div>")

    def _html(self):
        rows = "".join(self._row(r) for r in self.steps)
        return (_SHIM_CSS +
                f"<div style='font-family:{_UIFONT};max-width:560px;border:1px solid #ecebf3;"
                f"border-radius:14px;padding:15px 18px;background:#fff;"
                f"box-shadow:0 1px 5px rgba(20,20,40,.07)'>"
                f"<div style='font-weight:700;color:#1f2430;font-size:14.5px;"
                f"margin-bottom:8px'>{self.title}</div>{rows}</div>")

    def _r(self):
        try:
            update_display(HTML(self._html()), display_id=self.did)
        except Exception:
            pass


def _poll(cmd, ui, env=None):
    """Run a command, animating `ui` while it works; capture output for errors."""
    e = dict(os.environ, PYTHONUNBUFFERED="1")
    if env:
        e.update(env)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=e)
    buf = []
    while True:
        r, _, _ = select.select([p.stdout], [], [], 0.12)
        if r:
            ln = p.stdout.readline()
            if ln == "" and p.poll() is not None:
                break
            if ln.strip():
                buf.append(ln.rstrip("\n"))
        elif p.poll() is not None:
            break
        ui.tick()
    p.wait()
    return p.returncode, "\n".join(buf)


def _fetch_models(url, dest_models_dir):
    """Download the BindCORE LIP & MoRF weights into `dest_models_dir`. The URL may
    point at a .zip of the `models` directory or at a single file; both are handled.
    The two checkpoints are expected to end up at
    {dest_models_dir}/BindCORE_{LIP,MoRF}_IDPFold2/bindCORE.pt."""
    import zipfile, urllib.parse
    os.makedirs(dest_models_dir, exist_ok=True)
    parsed = urllib.parse.urlparse(url)
    is_zip = parsed.path.lower().endswith(".zip")
    dl = f"{CONTENT}/_bindcore_models_dl" + (".zip" if is_zip else ".bin")
    rc, out = _poll(f"wget -q -O '{dl}' '{url}'", ui)
    if rc or not os.path.exists(dl) or os.path.getsize(dl) == 0:
        raise SystemExit("model weights download failed:\n" + out[-1500:])
    # A zip download may actually be a zip even without a .zip suffix — sniff it.
    if zipfile.is_zipfile(dl):
        stage = f"{CONTENT}/_bindcore_models_stage"
        shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
        with zipfile.ZipFile(dl) as z:
            z.extractall(stage)
        # find every BindCORE_*_IDPFold2 dir and move it under dest_models_dir
        moved = 0
        for dp, dn, fn in os.walk(stage):
            if "__MACOSX" in dp:
                continue
            base = os.path.basename(dp)
            if base.startswith("BindCORE_") and "bindCORE.pt" in fn:
                tgt = os.path.join(dest_models_dir, base)
                shutil.rmtree(tgt, ignore_errors=True); shutil.move(dp, tgt)
                moved += 1
        if moved == 0:
            raise SystemExit("model zip extracted but no BindCORE_*/bindCORE.pt found.")
    else:
        # single-file download → place as the LIP checkpoint (adjust if needed)
        tgt = os.path.join(dest_models_dir, "BindCORE_LIP_IDPFold2")
        os.makedirs(tgt, exist_ok=True)
        shutil.move(dl, os.path.join(tgt, "bindCORE.pt"))


_STAMP = f"{CONTENT}/.bindcore_setup_done"
if not os.path.exists(_STAMP):
    ui = _Installer("⚙️  Setting up BindCORE · first run (~8–12 min, one time)")

    def step(label, cmd, env=None):
        rec = ui.begin(label)
        rc, out = _poll(cmd, ui, env)
        ui.end(rec, rc == 0)
        if rc:
            ui.close("✗ Setup failed at: " + label)
            print(out[-3500:]); raise SystemExit(label)

    step("Installing uv", "pip -q install uv")
    step("Analysis & visualization stack",
         f"{UVPIP} h5py mdtraj 'MDAnalysis>=2.8' pandas scikit-learn scipy matplotlib "
         f"plotly anywidget ipywidgets tqdm biopython pydantic PyYAML filelock")
    step("EnsembleMDP native deps (openmm · pdbfixer · prody)",
         f"{UVPIP} openmm pdbfixer prody")
    step("IDPFold2 dependencies",
         f"{UVPIP} einops loguru 'hydra-core>=1.3' omegaconf rootutils biotite "
         f"fair-esm torch_geometric")
    if not os.path.isdir(BINDCORE):
        step("Cloning BindCORE", f"git clone -q {BINDCORE_URL} {BINDCORE}")
    if not os.path.isdir(IDPFOLD2):
        step("Cloning IDPFold2", f"git clone -q {IDPFOLD2_URL} {IDPFOLD2}")
    step("Installing BindCORE & IDPFold2", f"{UVPIP} --no-deps -e {BINDCORE} -e {IDPFOLD2}")
    step("Installing EnsembleMDP", f"{UVPIP} --no-deps 'git+{ENSEMBLEMDP_URL}'")

    # download the LIP & MoRF model weights (the repo ships without them)
    _models_dir = f"{BINDCORE}/models"
    _lip_ckpt = f"{_models_dir}/BindCORE_LIP_IDPFold2/bindCORE.pt"
    if not os.path.exists(_lip_ckpt):
        _rec = ui.begin("BindCORE model weights (LIP & MoRF)")
        try:
            _fetch_models(BINDCORE_MODELS_URL, _models_dir)
            ui.end(_rec, os.path.exists(_lip_ckpt))
        except SystemExit as _e:
            ui.end(_rec, False)
            ui.close("✗ Setup failed: BindCORE model weights")
            print(_e); raise
        if not os.path.exists(_lip_ckpt):
            ui.close("✗ Setup failed: BindCORE model weights"); raise SystemExit("models")

    # background the weights download so it overlaps the slow cg2all build
    _wproc = None
    if not os.path.exists(IDP_CKPT):
        _wproc = subprocess.Popen(f"wget -q -O '{IDP_CKPT}' '{IDPFOLD2_CKPT_URL}'", shell=True)

    step("cg2all · isolated virtual env", f"uv venv {CG2ALL_ENV} --python 3.10 --seed")
    _VPY = f"{CG2ALL_ENV}/bin/python"
    step("cg2all · torch (CPU)",
         f"uv pip install --python {_VPY} torch==2.3.1 "
         f"--index-url https://download.pytorch.org/whl/cpu")
    # DGL: pin a CPU wheel that actually exists on the index (newest 403s)
    _rec = ui.begin("cg2all · DGL"); _dgl = None; _derr = ""
    for _v in ("2.4.0", "2.3.0", "2.2.1"):
        _rc, _out = _poll(f"uv pip install --python {_VPY} 'dgl=={_v}' -f {DGL_INDEX}", ui)
        if _rc == 0:
            _dgl = _v; break
        _derr = _out
    ui.end(_rec, _dgl is not None)
    if _dgl is None:
        ui.close("✗ Setup failed at: cg2all · DGL"); print(_derr[-3000:]); raise SystemExit("dgl")
    step("cg2all · package (compiles patched mdtraj — slow)",
         f"uv pip install --python {_VPY} '{CG2ALL_SPEC}' 'dgl=={_dgl}' -f {DGL_INDEX}")
    step("cg2all · model checkpoint",
         f"{_VPY} -c \"import cg2all.lib.libmodel as m; "
         f"from cg2all.lib.libconfig import MODEL_HOME; "
         f"p=MODEL_HOME/'CalphaBasedModel.ckpt'; "
         f"(not p.exists()) and m.download_ckpt_file('CalphaBasedModel', p)\"")

    _rec = ui.begin("IDPFold2 weights (downloaded in background)")
    if _wproc is not None:
        while _wproc.poll() is None:
            ui.tick(); time.sleep(0.12)
    ui.end(_rec, os.path.exists(IDP_CKPT))
    if not os.path.exists(IDP_CKPT):
        ui.close("✗ Setup failed: weights download"); raise SystemExit("weights")

    open(_STAMP, "w").write("ok")
    ui.close("✅  BindCORE ready — the studio is below")

try:
    from google.colab import output as _cwm
    _cwm.enable_custom_widget_manager()
except Exception:
    pass
for p in (BINDCORE, IDPFOLD2):
    if p not in sys.path:
        sys.path.insert(0, p)

# IDPFold2's torch-MoE counts tokens per expert with torch.histc, then runs
# repeat_interleave on that count. On CPU histc rejects int tensors and
# repeat_interleave rejects float ones — neither dtype works. Replace the whole
# expression with torch.bincount (the correct integer per-expert count), which works
# on CPU and GPU. Idempotent; handles the original and any earlier-patched form.
_moe = f"{IDPFOLD2}/src/model/components/moe_modules_torch.py"
try:
    if os.path.exists(_moe):
        _s = open(_moe).read()
        _new = "torch.bincount(top_expert.long(), minlength=self.n_experts)"
        for _old in ("torch.histc(top_expert, self.n_experts, 0, self.n_experts - 1)",
                     "torch.histc(top_expert.float(), self.n_experts, 0, self.n_experts - 1)"):
            _s = _s.replace(_old, _new)
        open(_moe, "w").write(_s)
except Exception as _e:
    print("IDPFold2 MoE patch skipped:", _e)

assert os.path.exists(CG2ALL_BIN), \
    "cg2all env missing — delete /content/.bindcore_setup_done and re-run."
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"Device: cuda · {torch.cuda.get_device_name(0)}")
else:
    print("Device: CPU  ⚠️  No GPU — IDPFold2 will work but be slow. For ~10× speed: "
          "Runtime → Change runtime type → T4 GPU, then re-run.")

import os, glob, time, subprocess, threading, select, re
import ipywidgets as _W
import numpy as np
import pandas as pd
import h5py
import torch
from pathlib import Path
from torch.utils.data import DataLoader

import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("mdtraj").setLevel(logging.ERROR)
logging.getLogger("MDAnalysis").setLevel(logging.ERROR)

from bindcore.engine.predictor import load_checkpoint
from bindcore.data.io import prepare_data
from bindcore.data.datasets import ProteinDataset, collate_proteins
from bindcore.data.properties_extraction import save_properties_to_h5
from EnsembleMDP.analysis.orchestrator import ProteinAnalyzer

MODELS = {
    "LIP":  f"{BINDCORE}/models/BindCORE_LIP_IDPFold2/bindCORE.pt",
    "MoRF": f"{BINDCORE}/models/BindCORE_MoRF_IDPFold2/bindCORE.pt",
}
_LOADED = {}


def get_model(kind):
    if kind not in _LOADED:
        _LOADED[kind] = load_checkpoint(MODELS[kind], torch.device(DEVICE))
    return _LOADED[kind]


# --------------------------------------------------------------------------
# Animated 4-stage checklist for one protein (main-thread, like the installer)
# --------------------------------------------------------------------------
class RunUI:
    _LBL = ("IDPFold2 · sampling {n} conformations",
            "cg2all · backmapping to all-atom",
            "EnsembleMDP · ensemble features",
            "BindCORE · LIP & MoRF prediction")

    def __init__(self, name, nconf):
        self.name = name; self.f = 0; self.hint = ""; self.footer = ""
        self.stages = [[self._LBL[i].format(n=nconf), "wait", 0.0, 0.0] for i in range(4)]
        self.cur = -1
        # A LIVE ipywidgets.HTML widget. NOT displayed here — on_run places `self.box`
        # directly into the run VBox. Updating `.value` mutates the same DOM node in
        # place (display()/Output stack duplicates in Colab; widget trees don't).
        self.box = _W.HTML(self._html())

    def start(self, i):
        self.cur = i; self.hint = ""
        self.stages[i][1] = "run"; self.stages[i][2] = time.time(); self._r()

    def done(self, i):
        self.stages[i][3] = time.time() - self.stages[i][2]
        self.stages[i][1] = "ok"; self.hint = ""; self._r()

    def fail(self, i):
        if 0 <= i < 4:
            self.stages[i][1] = "err"
        self._r()

    def tick(self, hint=None):
        self.f += 1
        if hint is not None:
            self.hint = hint
        self._r()

    def finish(self, out):
        self.footer = (
            f"<span style='color:#27ae60;font-weight:600'>✓ done in {out['_secs']:.0f}s</span>"
            f"<span style='color:#9aa3b2'> · "
            f"LIP {int(np.asarray(out['LIP']['binary']).sum())}/{out['length']} · "
            f"MoRF {int(np.asarray(out['MoRF']['binary']).sum())}/{out['length']} residues</span>")
        self._r()

    def _row(self, i):
        lbl, st, t0, dur = self.stages[i]
        hint = ""
        if st == "run":
            ic = f"<span style='color:#6c5ce7;font-size:15px'>{_SPIN[self.f % len(_SPIN)]}</span>"
            nm = f"<span class='bcsh' style='color:#3b2f63;font-weight:600'>{lbl}</span>"
            tm = f"{time.time() - t0:.0f}s"
            if self.hint:
                hint = (f"<div style='margin-left:28px;color:#aab0bd;font-size:11px;"
                        f"font-family:ui-monospace,Menlo,monospace;white-space:nowrap;"
                        f"overflow:hidden;text-overflow:ellipsis;max-width:500px'>"
                        f"{self.hint}</div>")
        elif st == "ok":
            ic = "<span style='color:#27ae60'>✓</span>"
            nm = f"<span style='color:#1f2430'>{lbl}</span>"; tm = f"{dur:.0f}s"
        elif st == "err":
            ic = "<span style='color:#e8633a'>✗</span>"
            nm = f"<span style='color:#e8633a;font-weight:600'>{lbl}</span>"; tm = ""
        else:
            ic = "<span style='color:#cfd4dd'>○</span>"
            nm = f"<span style='color:#aab0bd'>{lbl}</span>"; tm = ""
        return (f"<div style='display:flex;align-items:center;gap:11px;padding:3px 0'>"
                f"<span style='width:17px;text-align:center'>{ic}</span>"
                f"<span style='flex:1'>{nm}</span>"
                f"<span style='color:#9aa3b2;font-size:12px;"
                f"font-variant-numeric:tabular-nums'>{tm}</span></div>{hint}")

    def _html(self):
        rows = "".join(self._row(i) for i in range(4))
        foot = (f"<div style='margin-top:8px;padding-top:8px;border-top:1px solid #f1f1f6;"
                f"font-size:12.5px'>{self.footer}</div>" if self.footer else "")
        return (_SHIM_CSS +
                f"<div style='font-family:{_UIFONT};max-width:560px;border:1px solid #ecebf3;"
                f"border-radius:13px;padding:13px 16px;margin:6px 0;background:#fff;"
                f"box-shadow:0 1px 4px rgba(20,20,40,.06)'>"
                f"<div style='font-weight:700;color:#1f2430;margin-bottom:7px'>"
                f"\U0001f9ec {self.name}</div>{rows}{foot}</div>")

    def _r(self):
        try:
            self.box.value = self._html()
        except Exception:
            pass


def poll_run(cmd, ui, env=None):
    """Run a subprocess, animating `ui` (+ a one-line hint) until it finishes."""
    e = dict(os.environ, PYTHONUNBUFFERED="1")
    if env:
        e.update(env)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=e)
    buf = []; last = 0.0
    while True:
        r, _, _ = select.select([p.stdout], [], [], 0.12)
        if r:
            ln = p.stdout.readline()
            if ln == "" and p.poll() is not None:
                break
            ln = ln.rstrip("\n")
            if ln.strip():
                buf.append(ln)
        elif p.poll() is not None:
            break
        now = time.time()
        if now - last > 0.1:
            ui.tick(hint=(buf[-1][-74:] if buf else "")); last = now
    p.wait(); ui.tick()
    return p.returncode, "\n".join(buf)


def run_bg(fn, ui):
    """Run a blocking Python call in a worker thread while the main thread animates."""
    res = {}
    def w():
        try:
            res["v"] = fn()
        except Exception as e:  # noqa: BLE001
            res["e"] = e
    t = threading.Thread(target=w); t.start()
    while t.is_alive():
        ui.tick(); time.sleep(0.12)
    t.join(); ui.tick()
    if "e" in res:
        raise res["e"]
    return res["v"]


# --------------------------------------------------------------------------
# Pipeline steps
# --------------------------------------------------------------------------
def run_idpfold2(name, seq, nsamples, ui):
    workdir = os.path.join(WORK, name); os.makedirs(workdir, exist_ok=True)
    csv_path = os.path.join(workdir, "input.csv")
    pd.DataFrame([{"test_case": name, "sequence": seq}]).to_csv(csv_path, index=False)
    emb_dir = os.path.join(workdir, "embeddings"); os.makedirs(emb_dir, exist_ok=True)
    log_root = os.path.join(workdir, "idpfold_runs")
    # Batch many samples per forward pass: nsamples_per_pass = max_batch_length // nres.
    # Cap at 1500 residue-equivalents to stay within a 16 GB T4.
    mbl = min(len(seq) * int(nsamples), 1500)
    mbl = max(mbl, len(seq) + 1)
    cmd = (f"cd {IDPFOLD2} && OMP_NUM_THREADS=2 python -u src/inference.py "
           f"prefix={name} ckpt_dir='{IDP_CKPT}' plm_emb_dir='{emb_dir}' "
           f"csv_dir='{csv_path}' nsamples={int(nsamples)} max_batch_length={mbl} "
           f"logging_dir='{log_root}'")
    rc, out = poll_run(cmd, ui)
    cands = sorted(glob.glob(os.path.join(log_root, "**", "samples", f"{name}.pdb"),
                             recursive=True))
    if not cands:
        print("─── IDPFold2 log (tail) ───\n" + "\n".join(out.splitlines()[-25:]))
        raise RuntimeError(f"IDPFold2 produced no ensemble (exit {rc}).")
    return cands[-1]


def backmap(name, cg_pdb, ui):
    import mdtraj as md
    workdir = os.path.join(WORK, name, "aa"); os.makedirs(workdir, exist_ok=True)
    cg_top = os.path.join(workdir, "cg_top.pdb")
    cg_dcd = os.path.join(workdir, "cg_traj.dcd")
    t = md.load(cg_pdb)
    t[0].save_pdb(cg_top); t.save_dcd(cg_dcd)
    aa_pdb = os.path.join(workdir, "aa_topology.pdb")
    aa_dcd = os.path.join(workdir, "aa_traj.dcd")
    nfr = t.n_frames
    cmd = (f"OMP_NUM_THREADS=2 {CG2ALL_BIN} -p '{cg_top}' -d '{cg_dcd}' "
           f"-o '{aa_dcd}' -opdb '{aa_pdb}' --cg CalphaBasedModel "
           f"--device cpu --batch {min(nfr, 8)} --proc 2")
    rc, out = poll_run(cmd, ui)
    if not (os.path.exists(aa_dcd) and os.path.exists(aa_pdb)):
        print("─── cg2all log (tail) ───\n" + "\n".join(out.splitlines()[-25:]))
        raise RuntimeError(f"cg2all backmapping failed (exit {rc}).")
    aa_xtc = os.path.join(workdir, "aa_traj.xtc")
    md.load(aa_dcd, top=aa_pdb).save_xtc(aa_xtc)
    return aa_pdb, aa_xtc


def compute_features(name, aa_pdb, aa_xtc):
    analyzer = ProteinAnalyzer(Path(aa_pdb), Path(aa_xtc))
    props = analyzer.compute_all(sasa_n_sphere=960, contact_cutoff=8.0,
                                 scaling_min_sep=5)
    h5_path = os.path.join(WORK, name, "features.h5")
    if os.path.exists(h5_path):
        os.remove(h5_path)
    save_properties_to_h5({name: props}, h5_path)
    return h5_path, props


@torch.no_grad()
def predict(name, seq, h5_path, kind):
    model, ck = get_model(kind)
    df = pd.DataFrame([{"protein_id": name, "sequence": seq,
                        "LIP_annotations": "0" * len(seq)}])
    with h5py.File(h5_path, "r") as h5:
        Xs, Xl, Xp, seqs, _y, ids = prepare_data(
            df, h5, ck["scalar_features"], ck["local_features"],
            ck["pairwise_features"])
    ds = ProteinDataset(Xs, Xl, Xp, seqs, ids=ids, plm_h5_path=None)
    loader = DataLoader(ds, batch_size=1, shuffle=False, collate_fn=collate_proteins)
    dev = torch.device(DEVICE)
    for xs, xl, xp, sq, mask, _ids, plm in loader:
        logits = model(sq.long().to(dev), xs.to(dev), xl.to(dev),
                       xp.to(dev), mask.to(dev), None)
        if logits.dim() == 3 and logits.size(-1) == 1:
            logits = logits.squeeze(-1)
        probs = torch.sigmoid(logits)[0][mask[0].bool()].detach().cpu().numpy()
    thr = float(ck.get("best_threshold", 0.5))
    return probs.astype(float), (probs >= thr).astype(int), thr


def predict_all(name, seq, h5_path, props, aa_pdb):
    out = {"name": name, "seq": seq, "length": len(seq),
           "aa_pdb": aa_pdb, "props": props}
    for kind in ("LIP", "MoRF"):
        p, b, thr = predict(name, seq, h5_path, kind)
        out[kind] = {"prob": p, "binary": b, "thr": thr}
    return out


def _safe_id(name):
    """Filesystem/shell/Hydra-safe id for the pipeline. Protein names like
    'A0A0K2S4Q6|CD3CH_HUMAN' carry a '|' that the shell treats as a pipe (and
    other metacharacters break Hydra), so everything that touches a path or the
    IDPFold2 command line uses this slug. The original name is kept for display."""
    sid = re.sub(r"[^A-Za-z0-9._-]+", "_", str(name)).strip("_")
    return sid or "protein"


def analyse(name, seq, nsamples=20, ui=None, log=None):  # `log` ignored; `ui` is the
    seq = "".join(seq.split()).upper()                    # RunUI card on_run already
    sid = _safe_id(name)                                  # placed in the run VBox
    if ui is None:
        ui = RunUI(name, int(nsamples))
    t0 = time.time()
    try:
        ui.start(0); cg = run_idpfold2(sid, seq, nsamples, ui); ui.done(0)
        ui.start(1); aa_pdb, aa_xtc = backmap(sid, cg, ui); ui.done(1)
        ui.start(2); h5, props = run_bg(lambda: compute_features(sid, aa_pdb, aa_xtc), ui); ui.done(2)
        ui.start(3); out = run_bg(lambda: predict_all(sid, seq, h5, props, aa_pdb), ui); ui.done(3)
    except Exception:
        ui.fail(ui.cur)
        raise
    out["name"] = name              # keep the original name for the plots / report card
    out["_secs"] = time.time() - t0
    ui.finish(out)
    return out


def contiguous_regions(binary, min_len=3):
    binary = np.asarray(binary).astype(bool)
    regs, i, L = [], 0, len(binary)
    while i < L:
        if binary[i]:
            j = i
            while j < L and binary[j]:
                j += 1
            if j - i >= min_len:
                regs.append((i + 1, j))
            i = j
        else:
            i += 1
    return regs


print("BindCORE engine ready.")

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# --- design system --------------------------------------------------------
FONT     = "Inter, -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif"
FONT_CSS = "Inter, system-ui, -apple-system, Roboto, Helvetica, Arial, sans-serif"
INK   = "#1f2430"     # primary text
MUTED = "#8a93a6"     # secondary text / ticks
GRID  = "#eef1f6"     # gridlines
LIP_C  = "#2f6fed"    # LIP  — blue
MORF_C = "#e8633a"    # MoRF — warm orange
BRAND  = "#5b3fa6"    # indigo accent
PALETTE = ["#2f6fed", "#e8633a", "#13a8a0", "#a259c4", "#e0a800",
           "#d6336c", "#3aa655", "#5c7cfa"]


def _rgba(hexc, a):
    h = hexc.lstrip("#"); r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{a})"


def _show(fig):
    # Return a FigureWidget to be placed as a child widget — never display() it.
    # display()/Output render unreliably in Colab (duplicate cards); the studio puts
    # these straight into VBox containers instead.
    return go.FigureWidget(fig)


def _style(fig, height, title=None, subtitle=None, legend=False):
    fig.update_layout(
        template="plotly_white", height=height,
        font=dict(family=FONT, size=13, color=INK),
        paper_bgcolor="white", plot_bgcolor="white",
        margin=dict(t=82 if title else 28, b=48, l=66, r=30),
        showlegend=legend,
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0,
                    font=dict(size=12), bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(font=dict(family=FONT, size=12), bgcolor="white",
                        bordercolor=GRID))
    if title:
        t = f"<b>{title}</b>"
        if subtitle:
            t += (f"<br><span style='font-size:12.5px;color:{MUTED};"
                  f"font-weight:400'>{subtitle}</span>")
        fig.update_layout(title=dict(text=t, x=0.008, xanchor="left",
                                     y=0.97, yanchor="top",
                                     font=dict(size=17, color=INK)))
    fig.update_xaxes(showgrid=True, gridcolor=GRID, zeroline=False, linecolor=GRID,
                     ticks="outside", tickcolor=GRID, ticklen=5,
                     tickfont=dict(color=MUTED, size=11.5),
                     title_font=dict(color=MUTED, size=12.5))
    fig.update_yaxes(showgrid=True, gridcolor=GRID, zeroline=False,
                     linecolor="rgba(0,0,0,0)", tickfont=dict(color=MUTED, size=11.5),
                     title_font=dict(color=MUTED, size=12.5))
    return fig


def plot_tracks(s):
    """Per-residue LIP & MoRF binding propensity. Coloured y-axis labels name each
    track; the dotted line is the decision threshold; solid bars at the top mark
    contiguous predicted-binding regions."""
    L = s["length"]; x = np.arange(1, L + 1); aa = list(s["seq"])
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.13)
    counts = {}
    for r, kind, col in ((1, "LIP", LIP_C), (2, "MoRF", MORF_C)):
        d = s[kind]; p = np.asarray(d["prob"], float); thr = float(d["thr"])
        counts[kind] = int(np.asarray(d["binary"]).sum())
        fig.add_trace(go.Scatter(
            x=x, y=p, mode="lines", fill="tozeroy",
            line=dict(color=col, width=2.2, shape="spline", smoothing=0.5),
            fillcolor=_rgba(col, 0.12), customdata=aa,
            hovertemplate="res %{x} (%{customdata}) · p=%{y:.2f}<extra>" + kind + "</extra>"),
            r, 1)
        fig.add_hline(y=thr, line=dict(color=_rgba(col, 0.6), width=1.3, dash="dot"),
                      annotation_text=f"threshold {thr:.2f}",
                      annotation_position="top right",
                      annotation_font=dict(size=10, color=col), row=r, col=1)
        for a, b in contiguous_regions(d["binary"]):
            fig.add_shape(type="rect", x0=a - 0.5, x1=b + 0.5, y0=1.05, y1=1.13,
                          fillcolor=col, line_width=0, layer="above", row=r, col=1)
            fig.add_shape(type="rect", x0=a - 0.5, x1=b + 0.5, y0=0, y1=1.0,
                          fillcolor=_rgba(col, 0.06), line_width=0, layer="below",
                          row=r, col=1)
        fig.update_yaxes(range=[0, 1.18], tickvals=[0, 0.5, 1.0], title_text=kind,
                         title_font=dict(color=col, size=14), row=r, col=1)
    fig.update_xaxes(title_text="residue", row=2, col=1)
    _style(fig, 470,
           title=f"{s['name']} — per-residue binding prediction",
           subtitle=f"{L} aa · {counts['LIP']} LIP / {counts['MoRF']} MoRF residues "
                    "predicted binding · bars mark contiguous regions")
    return _show(fig)


def plot_fingerprint(s):
    """Conformational-ensemble fingerprint from EnsembleMDP scalar features: a Flory
    scaling-exponent gauge (folded → expanded) plus the global shape metrics."""
    pr = s["props"]
    def g(k):
        v = pr.get(k, float("nan"))
        return float(np.asarray(v).mean()) if np.ndim(v) else float(v)
    nu = g("scaling_exponent")
    ree2 = g("avg_squared_Ree")
    ree = ree2 ** 0.5 if (ree2 == ree2 and ree2 >= 0) else float("nan")
    nu_col = LIP_C if nu < 0.45 else ("#e0a800" if nu < 0.5 else MORF_C)
    fig = make_subplots(rows=1, cols=2, column_widths=[0.34, 0.66],
                        specs=[[{"type": "indicator"}, {"type": "xy"}]],
                        horizontal_spacing=0.20)
    fig.add_trace(go.Indicator(
        mode="gauge+number", value=nu,
        number=dict(valueformat=".3f", font=dict(size=26, color=INK)),
        gauge=dict(
            axis=dict(range=[0.30, 0.62], tickvals=[0.30, 0.40, 0.50, 0.60],
                      tickfont=dict(size=10, color=MUTED)),
            bar=dict(color=nu_col, thickness=0.32), bgcolor="white", borderwidth=0,
            steps=[dict(range=[0.30, 0.45], color="#eaf1fe"),
                   dict(range=[0.45, 0.50], color="#fff5dd"),
                   dict(range=[0.50, 0.62], color="#fceae3")],
            threshold=dict(line=dict(color=INK, width=2), thickness=0.8, value=0.50))),
        1, 1)
    # (label, value, unit, display-reference max) — bar length is value/ref so the
    # Å distances and the dimensionless shape descriptors read on one uniform scale;
    # the text label always shows the true measured value.
    metrics = [("Radius of gyration", g("radius_of_gyration_mean"), "Å", 60.0),
               ("End-to-end √⟨Ree²⟩", ree, "Å", 120.0),
               ("Max diameter", g("avg_maximum_diameter"), "Å", 200.0),
               ("Asphericity", g("asphericity_mean"), "", 1.0),
               ("Shape anisotropy", g("rel_shape_anisotropy_mean"), "", 1.0),
               ("Prolateness", g("prolateness_mean"), "", 1.0)][::-1]
    names = [m[0] for m in metrics]
    bar = [float(np.clip(m[1] / m[3] if m[1] == m[1] else 0.0, 0, 1)) for m in metrics]
    txt = [(f" {v:.1f} {u}".rstrip() if u else f" {v:.2f}") for _, v, u, _ in metrics]
    fig.add_trace(go.Bar(
        x=bar, y=names, orientation="h", marker=dict(color=BRAND),
        text=txt, textposition="outside", textfont=dict(size=11.5, color=INK),
        cliponaxis=False, hovertemplate="%{y}<extra></extra>"), 1, 2)
    _style(fig, 360,
           title=f"{s['name']} — conformational-ensemble fingerprint",
           subtitle="Flory ν: folded ≈0.33 · ideal-chain 0.50 (marker) · "
                    "expanded / IDP ≈0.6   |   bars scaled to typical ranges")
    fig.update_xaxes(visible=False, range=[0, 1.25], row=1, col=2)
    fig.update_yaxes(showgrid=False, automargin=True, row=1, col=2)
    fig.update_layout(margin=dict(t=84, b=36, l=28, r=66))
    fig.add_annotation(xref="paper", yref="paper", x=0.17, y=-0.10, showarrow=False,
                       text=f"<span style='color:{MUTED};font-size:11px'>"
                            "scaling exponent ν</span>", xanchor="center")
    return _show(fig)


def plot_overlay(samples, kind="LIP"):
    """Length-normalised per-residue propensity, one line per protein."""
    base = LIP_C if kind == "LIP" else MORF_C
    fig = go.Figure()
    for i, s in enumerate(samples):
        p = np.asarray(s[kind]["prob"], float); xn = np.linspace(0, 1, len(p))
        c = PALETTE[i % len(PALETTE)] if len(samples) > 1 else base
        fig.add_trace(go.Scatter(
            x=xn, y=p, mode="lines", name=s["name"],
            line=dict(color=c, width=2.2, shape="spline", smoothing=0.5),
            hovertemplate=s["name"] + " · %{y:.2f}<extra></extra>"))
    _style(fig, 430, legend=len(samples) > 1,
           title=f"{kind} propensity overlay",
           subtitle="length-normalised · 0 = N-terminus, 1 = C-terminus")
    fig.update_xaxes(title_text="relative position", range=[0, 1])
    fig.update_yaxes(title_text="probability", range=[0, 1.02])
    return _show(fig)


def plot_landscape(samples):
    """One marker per protein: x = length, y = % LIP residues, size ∝ % MoRF."""
    xs = [s["length"] for s in samples]
    ys = [100 * float(np.asarray(s["LIP"]["binary"]).mean()) for s in samples]
    ms = [100 * float(np.asarray(s["MoRF"]["binary"]).mean()) for s in samples]
    txt = [s["name"] for s in samples]
    fig = go.Figure(go.Scatter(
        x=xs, y=ys, mode="markers+text", text=txt, textposition="top center",
        textfont=dict(size=11, color=INK),
        marker=dict(size=[14 + 0.45 * m for m in ms], color=ys, colorscale="Tealrose",
                    cmin=0, cmax=max(ys + [1]), showscale=True,
                    colorbar=dict(title=dict(text="% LIP", font=dict(size=11)),
                                  thickness=12, outlinewidth=0,
                                  tickfont=dict(size=10, color=MUTED)),
                    line=dict(width=1.5, color="white"), opacity=0.95),
        customdata=ms,
        hovertemplate="<b>%{text}</b><br>length %{x} aa<br>LIP %{y:.0f}%<br>"
                      "MoRF %{customdata:.0f}%<extra></extra>"))
    _style(fig, 460, title="Binding landscape",
           subtitle="one marker per protein · marker size ∝ % MoRF residues")
    fig.update_xaxes(title_text="length (aa)")
    fig.update_yaxes(title_text="% residues predicted LIP")
    return _show(fig)


def report_card(s):
    def pct(k): return 100 * float(np.asarray(s[k]["binary"]).mean())
    def nreg(k): return len(contiguous_regions(s[k]["binary"]))
    pr = s["props"]
    nu = float(np.asarray(pr.get("scaling_exponent", float("nan"))).mean()) \
        if "scaling_exponent" in pr else float("nan")
    rg = float(np.asarray(pr.get("radius_of_gyration_mean", float("nan"))).mean()) \
        if "radius_of_gyration_mean" in pr else float("nan")
    regs = contiguous_regions(s["LIP"]["binary"])
    reg_str = ", ".join(f"{a}–{b}" for a, b in regs[:8]) or "none ≥3 aa"

    def chip(v, c):
        return (f"<span style='background:{_rgba(c, 0.14)};color:{c};padding:2px 9px;"
                f"border-radius:999px;font-weight:600;font-size:12.5px'>{v}</span>")

    def sub(t):
        return f"<span style='color:{MUTED};font-size:12px'>{t}</span>"

    rows = [
        ("Length", f"{s['length']} aa"),
        ("LIP residues", chip(f"{pct('LIP'):.0f}%", LIP_C) + " " + sub(f"· {nreg('LIP')} regions")),
        ("MoRF residues", chip(f"{pct('MoRF'):.0f}%", MORF_C) + " " + sub(f"· {nreg('MoRF')} regions")),
        ("Flory scaling ν", f"{nu:.3f}"),
        ("Radius of gyration", f"{rg:.1f} Å"),
        ("LIP binding regions",
         f"<span style='font-family:ui-monospace,SFMono-Regular,Menlo,monospace;"
         f"font-size:12.5px'>{reg_str}</span>"),
    ]
    body = "".join(
        f"<div style='display:flex;justify-content:space-between;align-items:center;"
        f"gap:18px;padding:9px 16px;background:{'#faf9fd' if i % 2 else '#ffffff'}'>"
        f"<span style='color:{MUTED};font-size:13px'>{k}</span>"
        f"<span style='color:{INK};font-weight:600;font-size:13px;text-align:right'>"
        f"{v}</span></div>"
        for i, (k, v) in enumerate(rows))
    html = (
        f"<div style='font-family:{FONT_CSS};max-width:540px;border:1px solid #ecebf3;"
        f"border-radius:14px;overflow:hidden;box-shadow:0 1px 4px rgba(20,20,40,.07);"
        f"margin:4px 0 10px'>"
        f"<div style='background:linear-gradient(90deg,{BRAND},#7b5fd1);color:#fff;"
        f"padding:13px 16px;display:flex;justify-content:space-between;"
        f"align-items:center'>"
        f"<span style='font-weight:700;font-size:15px'>\U0001f9ec {s['name']}</span>"
        f"<span style='font-weight:500;font-size:12px;opacity:.85'>BindCORE report</span>"
        f"</div>{body}</div>")
    return _W.HTML(html)


print("Plots loaded.")

import ipywidgets as W
from IPython.display import display, clear_output, HTML
from tqdm.auto import tqdm

SAMPLES = {}          # name -> result dict (after analyse)
QUEUE   = {}          # name -> {"seq":..., "status":...}

EXAMPLES = {
    "alpha_synuclein": "MDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLYVGSKTKEGVVHGVATVAEKTKEQVTNVGGAVVTGVTAVAQKTVEGAGSIAAATGFVKKDQLGKNEEGAPQEGILEDMPVDPDNEAYEMPSEEGYQDYEPEA",
}


def _card(title, desc, *widgets):
    head = W.HTML(f"<div style='font-weight:700;color:#3b1d5e;font-size:15px'>{title}</div>"
                  f"<div style='color:#666;font-size:12px;margin-bottom:6px'>{desc}</div>")
    return W.VBox([head, *widgets],
                  layout=W.Layout(border="1px solid #e4ddef", border_radius="10px",
                                  padding="12px", margin="6px 0"))


# --- input panel ----------------------------------------------------------
fasta_in = W.Textarea(
    placeholder=(">sp|P37840|SYUA_HUMAN Alpha-synuclein\n"
                 "MDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLY...\n"
                 ">my_peptide\n"
                 "GSHMSTAVLENPGL...\n\n"
                 "Paste one or more FASTA records — or just a raw sequence."),
    layout=W.Layout(width="98%", height="150px"),
    style={"description_width": "0px"})
nconf   = W.IntSlider(description="conformations", value=20, min=5, max=100, step=5,
                      continuous_update=False, layout=W.Layout(width="360px"))
add_btn = W.Button(description="➕ Add to queue", button_style="info")
ex_btn  = W.Button(description="Load example")
# A live HTML widget (NOT an Output): clear_output() inside an ipywidgets.Output does
# not reliably clear in Colab, so re-rendering the queue stacked duplicate tables.
queue_box = W.HTML()


def parse_fasta(text):
    """Parse FASTA text into [(name, sequence), …]. The name is the first token of
    each header; a raw sequence with no '>' header is accepted as a single record."""
    text = (text or "").strip()
    if not text:
        return []
    if not text.startswith(">"):
        seq = "".join(c for c in text.upper() if c.isalpha())
        return [("protein", seq)] if seq else []
    recs, name, chunks = [], None, []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if name is not None:
                recs.append((name, "".join(chunks)))
            hdr = line[1:].strip()
            name = (hdr.split()[0] if hdr.split() else "") or f"seq_{len(recs) + 1}"
            chunks = []
        else:
            chunks.append("".join(c for c in line.upper() if c.isalpha()))
    if name is not None:
        recs.append((name, "".join(chunks)))
    return [(n, s) for n, s in recs if s]


def render_queue():
    if not QUEUE:
        queue_box.value = ("<div style='font-family:sans-serif;font-size:13px;color:#888;"
                           "padding:4px 2px'>Queue empty — add a protein above.</div>")
        return
    h = ["<table style='font-family:sans-serif;font-size:13px;border-collapse:collapse'>"
         "<tr><th style='text-align:left;padding:4px 10px'>Protein</th>"
         "<th style='padding:4px 10px'>Len</th>"
         "<th style='padding:4px 10px'>Status</th></tr>"]
    for nm, q in QUEUE.items():
        color = {"queued": "#888", "running": "#fe9929",
                 "done": "#06a77d", "error": "#d7301f"}.get(q["status"], "#888")
        h.append(f"<tr><td style='padding:4px 10px'>{nm}</td>"
                 f"<td style='padding:4px 10px;text-align:center'>{len(q['seq'])}</td>"
                 f"<td style='padding:4px 10px;color:{color};font-weight:600'>"
                 f"{q['status']}</td></tr>")
    h.append("</table>")
    queue_box.value = "".join(h)


def on_add(_):
    recs = parse_fasta(fasta_in.value)
    if not recs:
        queue_box.value = ("<div style='font-family:sans-serif;font-size:13px;color:#d7301f;"
                           "padding:4px 2px'>Paste a FASTA record (&gt;header + sequence) "
                           "or a raw sequence.</div>")
        return
    for nm, seq in recs:
        key, n = nm, 2                       # keep queue keys unique
        while key in QUEUE:
            key = f"{nm}_{n}"; n += 1
        QUEUE[key] = {"seq": seq, "status": "queued"}
    fasta_in.value = ""
    render_queue()


def on_example(_):
    fasta_in.value = ">sp|P37840|SYUA_HUMAN Alpha-synuclein\n" + EXAMPLES["alpha_synuclein"]


add_btn.on_click(on_add); ex_btn.on_click(on_example)


# --- run panel ------------------------------------------------------------
run_btn = W.Button(description="🚀 Run BindCORE", button_style="success",
                   layout=W.Layout(width="220px"))
run_box = W.VBox([])


def on_run(_):
    todo = [nm for nm, q in QUEUE.items() if q["status"] in ("queued", "error")]
    if not todo:                       # nothing to do — keep existing results
        return
    run_btn.disabled = True            # block re-clicks while a run is in progress
    run_box.children = ()              # fresh widget tree (no display()/Output/tqdm)
    try:
        ok = 0; t0 = time.time()
        for nm in todo:
            QUEUE[nm]["status"] = "running"; render_queue()
            ui = RunUI(nm, nconf.value)                       # an HTML widget…
            run_box.children = run_box.children + (ui.box,)   # …placed, not displayed
            try:
                SAMPLES[nm] = analyse(nm, QUEUE[nm]["seq"], nconf.value, ui=ui)
                QUEUE[nm]["status"] = "done"; ok += 1
            except Exception as e:
                ui.fail(ui.cur)
                run_box.children = run_box.children + (W.HTML(
                    f"<div style='color:#d7301f;font-family:sans-serif;font-size:13px;"
                    f"padding:2px 4px'>✗ {nm}: {e}</div>"),)
                QUEUE[nm]["status"] = "error"
            render_queue()
        run_box.children = run_box.children + (W.HTML(
            f"<div style='color:#27ae60;font-weight:600;font-family:sans-serif;"
            f"font-size:13px;padding:6px 4px 0'>✓ Finished {ok}/{len(todo)} "
            f"protein(s) in {time.time() - t0:.0f}s.</div>"),)
        result_sel.options = list(SAMPLES.keys())
        if SAMPLES and not result_sel.value:
            result_sel.value = list(SAMPLES.keys())[0]
    finally:
        run_btn.disabled = False


run_btn.on_click(on_run)


# --- explore panel --------------------------------------------------------
result_sel = W.Dropdown(description="Sample", options=[], layout=W.Layout(width="320px"))
show_btn   = W.Button(description="🔬 Explore", button_style="primary")
result_box = W.VBox([])


def on_show(_):
    nm = result_sel.value
    if not nm or nm not in SAMPLES:
        return
    s = SAMPLES[nm]
    # Pure widget tree: report card (HTML widget) + two FigureWidgets, set as the
    # panel's children. No display()/Output, so nothing ever stacks.
    result_box.children = (report_card(s), plot_tracks(s), plot_fingerprint(s))


show_btn.on_click(on_show)


# --- compare panel --------------------------------------------------------
cmp_btn = W.Button(description="📊 Compare all", layout=W.Layout(width="200px"))
cmp_kind = W.ToggleButtons(options=["LIP", "MoRF"], value="LIP")
cmp_box = W.VBox([])


def on_compare(_):
    samples = list(SAMPLES.values())
    if not samples:
        cmp_box.children = (W.HTML("<div style='font-family:sans-serif;font-size:13px;"
                                   "color:#888'>Run at least one protein first.</div>"),)
        return
    cmp_box.children = (plot_landscape(samples), plot_overlay(samples, cmp_kind.value))


cmp_btn.on_click(on_compare)


# --- layout ---------------------------------------------------------------
panel_in = _card("1 · Add proteins",
                 "Paste one or more sequences in FASTA format (>header + sequence); "
                 "every record is queued. Choose how many conformations IDPFold2 "
                 "samples, then add to the queue.",
                 fasta_in, nconf, W.HBox([add_btn, ex_btn]), queue_box)
panel_run = _card("2 · Run the pipeline",
                  "Sequence → IDPFold2 ensemble → cg2all backmapping → "
                  "EnsembleMDP features → BindCORE LIP & MoRF prediction.",
                  W.HBox([run_btn]), run_box)
panel_res = _card("3 · Explore a protein",
                  "Report card, per-residue LIP & MoRF tracks, and the "
                  "conformational-ensemble fingerprint.",
                  W.HBox([result_sel, show_btn]), result_box)
panel_cmp = _card("4 · Compare proteins",
                  "Binding landscape and a length-normalised propensity overlay.",
                  W.HBox([cmp_kind, cmp_btn]), cmp_box)

display(W.HTML(
    "<h2 style='font-family:sans-serif;color:#3b1d5e;margin-bottom:0'>"
    "🧬 BindCORE Studio</h2>"
    "<div style='font-family:sans-serif;color:#666;margin-bottom:8px'>"
    "Per-residue LIP &amp; MoRF binding prediction from sequence, via an "
    "IDPFold2 conformational ensemble.</div>"))
display(panel_in, panel_run, panel_res, panel_cmp)
render_queue()